In [50]:
# Basic stuff
import pandas as pd

#movies_df = pd.read_csv("movie_dataset.csv")

### Model that predicts whether a movie will receive an Oscar nomination, and then study how much the movie’s budget contributes to that prediction.

## Cleaning TMDB

In [51]:
# Reading file
tmdb_raw = pd.read_csv(
    "TMDB_all_movies.csv",
    usecols=[
        "id",
        "imdb_id",
        "title",
        "original_title",
        "release_date",
        "runtime",
        "original_language",
        "budget",
        "genres",
        "status"
    ]
)

In [52]:
tmdb_df = tmdb_raw.copy()

# Keep released movies
tmdb_df = tmdb_df[tmdb_df["status"] == "Released"].copy()

# Convert release date
release_dates = pd.to_datetime(
    tmdb_df["release_date"],
    errors="coerce"
)

tmdb_df["release_year"] = release_dates.dt.year
tmdb_df["release_month"] = release_dates.dt.month

# Year filter
tmdb_df = tmdb_df[
    (tmdb_df["release_year"] >= 1980) &
    (tmdb_df["release_year"] <= 2025)
].copy()

# Keep only useful columns
tmdb_df = tmdb_df[
    [
        "id",
        "imdb_id",
        "title",
        "original_title",
        "release_year",
        "release_month",
        "runtime",
        "original_language",
        "budget",
        "genres"
    ]
].copy()

In [53]:
print("Total movies:", len(tmdb_df))

print("\nMissing values:")
print(
    tmdb_df.isna()
        .sum()
        .sort_values(ascending=False)
)

print("\nMissing IMDb IDs:", tmdb_df["imdb_id"].isna().sum())

print(
    "Duplicate non-null IMDb IDs:",
    tmdb_df.loc[tmdb_df["imdb_id"].notna(), "imdb_id"]
        .duplicated()
        .sum()
)

print("Zero budgets:", (tmdb_df["budget"] == 0).sum())
print("Zero runtimes:", (tmdb_df["runtime"] == 0).sum())

print("\nYear range:")
print(tmdb_df["release_year"].min())
print(tmdb_df["release_year"].max())

Total movies: 874194

Missing values:
imdb_id              393507
genres               204977
title                     5
original_title            5
id                        0
release_year              0
release_month             0
runtime                   0
original_language         0
budget                    0
dtype: int64

Missing IMDb IDs: 393507
Duplicate non-null IMDb IDs: 0
Zero budgets: 810596
Zero runtimes: 173363

Year range:
1980.0
2025.0


In [54]:
# Replace fake zeros
tmdb_df["budget"] = tmdb_df["budget"].replace(0, pd.NA)
tmdb_df["runtime"] = tmdb_df["runtime"].replace(0, pd.NA)

print("Movies with budget:", tmdb_df["budget"].notna().sum())
print("Movies with runtime:", tmdb_df["runtime"].notna().sum())
print("Movies with genres:", tmdb_df["genres"].notna().sum())

# Remove title missing rows (5 rows missing only)
tmdb_df = tmdb_df.dropna(subset=["title"])

print("Movies with IMDb ID:", tmdb_df["imdb_id"].notna().sum())

Movies with budget: 63598
Movies with runtime: 700831
Movies with genres: 669217
Movies with IMDb ID: 480685


In [55]:
# Delete raw versions
del tmdb_raw

## Cleaning IMDB

In [56]:
imdb_raw = pd.read_csv(
    "title.basics.tsv",
    sep="\t",
    na_values="\\N",
    usecols=[
        "tconst",
        "titleType",
        "primaryTitle",
        "originalTitle",
        "startYear",
        "genres"
    ]
)

In [57]:
imdb_df = imdb_raw.copy()

imdb_df = imdb_df[imdb_df["titleType"] == "movie"].copy()

imdb_df = imdb_df[
    [
        "tconst",
        "primaryTitle",
        "originalTitle",
        "startYear",
        "genres"
    ]
].copy()

In [58]:
# Rename
imdb_df = imdb_df.rename(columns={
    "tconst": "imdb_id",
    "primaryTitle": "imdb_title",
    "originalTitle": "imdb_original_title",
    "startYear": "imdb_year",
    "genres": "imdb_genres"
})

# Convert year
imdb_df["imdb_year"] = pd.to_numeric(
    imdb_df["imdb_year"],
    errors="coerce"
)

# Dropping missing titles
imdb_df = imdb_df.dropna(subset=["imdb_title"])

# Year range
imdb_df = imdb_df[
    (imdb_df["imdb_year"] >= 1980) &
    (imdb_df["imdb_year"] <= 2025)
].copy()

print(imdb_df.shape)
print(imdb_df.isna().sum())

print("Duplicate IMDb IDs:", imdb_df["imdb_id"].duplicated().sum())

(450567, 5)
imdb_id                    0
imdb_title                 0
imdb_original_title        0
imdb_year                  0
imdb_genres            33223
dtype: int64
Duplicate IMDb IDs: 0


In [59]:
# Delete raw version
del imdb_raw

## Combining datas (Becasue TMDB doesnt show which are movies and so we combine from IMDB id with TMDB id so that this way only movies are left)

In [60]:
combined_df = tmdb_df.merge(
    imdb_df,
    on="imdb_id",
    how="left"
)

# Leave only confirmed IMDB movies
valid_movie_ids = set(imdb_df["imdb_id"])

combined_df = combined_df[
    combined_df["imdb_id"].isin(valid_movie_ids)
].copy()

print("Confirmed IMDb movies:", combined_df.shape)

# Delete duplicate rows from IMDB
combined_df = combined_df.drop(columns=[
    "imdb_title",
    "imdb_original_title",
    "imdb_year"
])

Confirmed IMDb movies: (253107, 14)


In [61]:
combined_df = combined_df.drop(
    columns=[
        "imdb_title",
        "imdb_original_title",
        "imdb_year",
        "imdb_genres"
    ],
    errors="ignore"
)

In [62]:
print(combined_df.shape)
print(combined_df.isna().sum())

(253107, 10)
id                        0
imdb_id                   0
title                     0
original_title            0
release_year              0
release_month             0
runtime               26772
original_language         0
budget               226371
genres                24616
dtype: int64


## Cleaning Oscar data

In [63]:
oscars_raw = pd.read_csv(
    "full_data.csv",
    sep="\t",
    usecols=[
        "Year",
        "Class",
        "CanonicalCategory",
        "Category",
        "Film",
        "FilmId",
        "Winner"
    ]
)

In [64]:
oscars_df = oscars_raw.copy()

# Filter by class == Title only
oscars_df = oscars_df[
    oscars_df["Class"] == "Title"
].copy()

oscars_df = oscars_df[
    [
        "Year",
        "Class",
        "CanonicalCategory",
        "Category",
        "Film",
        "FilmId",
        "Winner"
    ]
].copy()

# Convert real Year only the year
oscars_df["oscar_year"] = pd.to_numeric(
    oscars_df["Year"].astype(str).str[:4],
    errors="coerce"
)

# Filter by year
oscars_df = oscars_df[
    (oscars_df["oscar_year"] >= 1980) &
    (oscars_df["oscar_year"] <= 2025)
].copy()

# Convert missing values to False
oscars_df["Winner"] = (
    oscars_df["Winner"]
    .fillna(False)
    .astype(bool)
)

# Some movies have multiples IDs so split them
oscars_df["Film"] = oscars_df["Film"].str.split("|")
oscars_df["FilmId"] = oscars_df["FilmId"].str.split("|")

oscars_df = oscars_df.explode(
    ["Film", "FilmId"],
    ignore_index=True
)

# Rename
oscars_df = oscars_df.rename(columns={
    "Film": "oscar_film",
    "FilmId": "imdb_id"
})

# Clean white space
oscars_df["oscar_film"] = oscars_df["oscar_film"].str.strip()
oscars_df["imdb_id"] = oscars_df["imdb_id"].str.strip()

In [65]:
# Validate
print(
    oscars_df["imdb_id"]
    .dropna()
    .str.match(r"^tt\d+$")
    .value_counts()
)

oscar_ids = set(oscars_df["imdb_id"].dropna())
movie_ids = set(combined_df["imdb_id"].dropna())

matched_ids = oscar_ids & movie_ids

print("Unique Oscar IMDb IDs:", len(oscar_ids))
print("Matched to movie dataset:", len(matched_ids))
print(
    "Match percentage:",
    len(matched_ids) / len(oscar_ids) * 100
)

imdb_id
True    1496
Name: count, dtype: int64
Unique Oscar IMDb IDs: 1477
Matched to movie dataset: 838
Match percentage: 56.73662830060935


In [66]:
# Delete raw
del oscars_raw

## Oscars per category nominations

In [67]:
print(
    oscars_df["CanonicalCategory"]
    .value_counts()
    .to_string()
)

CanonicalCategory
BEST PICTURE                   301
DOCUMENTARY (Feature)          230
INTERNATIONAL FEATURE FILM     230
DOCUMENTARY (Short Subject)    213
SHORT FILM (Live Action)       210
SHORT FILM (Animated)          203
ANIMATED FEATURE FILM          109


In [68]:
# Leave only those 4 categories
feature_film_categories = {
    "BEST PICTURE": "best_picture",
    "ANIMATED FEATURE FILM": "animated_feature",
    "INTERNATIONAL FEATURE FILM": "international_feature",
    "DOCUMENTARY (Feature)": "documentary_feature"
}

oscars_df = oscars_df[
    oscars_df["CanonicalCategory"].isin(feature_film_categories)
].copy()

# Start with one row per Oscar movie
oscar_targets = (
    oscars_df[["imdb_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# General targets

# Group all Oscar rows belonging to the same movie.
general = (
    oscars_df
    .groupby("imdb_id")
    .agg(
        any_feature_film_nomination=("Winner", "size"),
        any_feature_film_win=("Winner", "sum")
    )
    .reset_index()
)

general["any_feature_film_nomination"] = (
    general["any_feature_film_nomination"] > 0
).astype(int)

general["any_feature_film_win"] = (
    general["any_feature_film_win"] > 0
).astype(int)

general = general[
    [   
        "imdb_id",
        "any_feature_film_nomination",
        "any_feature_film_win"
    ]
]

# Combine
oscar_targets = oscar_targets.merge(
    general,
    on="imdb_id",
    how="left"
)

In [69]:
for category, prefix in feature_film_categories.items():

    category_df = oscars_df[
        oscars_df["CanonicalCategory"] == category
    ]

    # Group all category rows belonging to the same movie.
    summary = (
        category_df
        .groupby("imdb_id")
        .agg(
            nomination=("Winner", "size"),
            win=("Winner", "sum")
        )
        .reset_index()
    )

    summary[f"{prefix}_nomination"] = (
        summary["nomination"] > 0
    ).astype(int)

    summary[f"{prefix}_win"] = (
        summary["win"] > 0
    ).astype(int)

    summary = summary[
        [
            "imdb_id",
            f"{prefix}_nomination",
            f"{prefix}_win"
        ]
    ]

    oscar_targets = oscar_targets.merge(
        summary,
        on="imdb_id",
        how="left"
    )

In [70]:
# Not nominated == 0
# Nominated/Won == 1
target_cols = [
    col for col in oscar_targets.columns
    if col != "imdb_id"
]

oscar_targets[target_cols] = (
    oscar_targets[target_cols]
    .fillna(0)
    .astype(int)
)

In [71]:
print(oscar_targets.shape)
#print(oscar_targets.head())
print(oscar_targets.sum(numeric_only=True))

(851, 11)
any_feature_film_nomination         851
any_feature_film_win                163
best_picture_nomination             301
best_picture_win                     46
animated_feature_nomination         109
animated_feature_win                 25
international_feature_nomination    230
international_feature_win            46
documentary_feature_nomination      230
documentary_feature_win              47
dtype: int64


## Merge oscars and movies

In [72]:
final_df = combined_df.merge(
    oscar_targets,
    on="imdb_id",
    how="left"
)

target_cols = [
    col for col in final_df.columns
    if col.endswith("_nomination")
    or col.endswith("_win")
]

final_df[target_cols] = (
    final_df[target_cols]
    .fillna(0)
    .astype(int)
)

# Conver year to nullable integer
final_df["release_year"] = final_df["release_year"].astype("Int64")

# Remove the same columns they are not needed
final_df = final_df.drop(
    columns=[
        "original_title",
        "imdb_genres"
    ],
    errors="ignore"
)

In [73]:
print(final_df.shape)

print(
    final_df["any_feature_film_nomination"]
    .value_counts()
)

(253107, 19)
any_feature_film_nomination
0    252275
1       832
Name: count, dtype: int64


## Inflation adjustment (since 20mil in 1980 is not the same as in 2025)

In [74]:
# annual CPI-U values from the U.S. Bureau of Labor Statistics
cpi = {
    1980: 82.4,   1981: 90.9,   1982: 96.5,   1983: 99.6,
    1984: 103.9,  1985: 107.6,  1986: 109.6,  1987: 113.6,
    1988: 118.3,  1989: 124.0,  1990: 130.7,  1991: 136.2,
    1992: 140.3,  1993: 144.5,  1994: 148.2,  1995: 152.4,
    1996: 156.9,  1997: 160.5,  1998: 163.0,  1999: 166.6,
    2000: 172.2,  2001: 177.1,  2002: 179.9,  2003: 184.0,
    2004: 188.9,  2005: 195.3,  2006: 201.6,  2007: 207.342,
    2008: 215.303, 2009: 214.537, 2010: 218.056, 2011: 224.939,
    2012: 229.594, 2013: 232.957, 2014: 236.736, 2015: 237.017,
    2016: 240.007, 2017: 245.120, 2018: 251.107, 2019: 255.657,
    2020: 258.811, 2021: 270.970, 2022: 292.655, 2023: 304.702,
    2024: 313.689, 2025: 321.943
}

# Match each movie year to CPI
final_df["cpi"] = final_df["release_year"].map(cpi)

CPI_2025 = 321.943

# Make sure budget is numeric
final_df["budget"] = pd.to_numeric(
    final_df["budget"],
    errors="coerce"
)

final_df["cpi"] = pd.to_numeric(
    final_df["cpi"],
    errors="coerce"
)

# Adjust budget to 2025 dollars
final_df["budget_2025"] = (
    final_df["budget"] *
    (CPI_2025 / final_df["cpi"])
)

# Round values
final_df["budget_2025"] = (
    final_df["budget_2025"]
    .round()
    .astype("Int64")
)

final_df["budget"] = (
    final_df["budget"]
    .round()
    .astype("Int64")
)

In [75]:
print(
    final_df[
        [
            "title",
            "release_year",
            "budget",
            "budget_2025",
        ]
    ].dropna().head(20)
)

                                                title  release_year  \
2                                          Four Rooms          1995   
3                                      Judgment Night          1993   
4                    Life in Loops (A Megacities RMX)          2006   
5                                        Finding Nemo          2003   
6                                        Forrest Gump          1994   
7                                     American Beauty          1999   
8                                  Dancer in the Dark          2000   
10                                  The Fifth Element          1997   
11                                 My Life Without Me          2003   
12  Pirates of the Caribbean: The Curse of the Bla...          2003   
13                                  Kill Bill: Vol. 1          2003   
14                                            Jarhead          2005   
15                                      Walk on Water          2004   
16    

## Encode genres and languages

In [76]:
genre_dummies = final_df["genres"].str.get_dummies(sep=", ")

final_df = pd.concat(
    [final_df, genre_dummies],
    axis=1
)

print(genre_dummies.columns.tolist())

['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']


In [77]:
# Keep only languages with more than 1000 movies
language_counts = final_df["original_language"].value_counts()

common_languages = language_counts[
    language_counts >= 1000
].index

final_df["language_grouped"] = final_df["original_language"].where(
    final_df["original_language"].isin(common_languages),
    "Other"
)

language_dummies = pd.get_dummies(
    final_df["language_grouped"],
    prefix="lang",
    dtype=int
)

final_df = pd.concat(
    [final_df, language_dummies],
    axis=1
)

print(final_df["language_grouped"].value_counts())

language_grouped
en       105085
es        17674
Other     13782
fr        13273
ja        12732
de         8962
zh         7501
it         7032
pt         5738
ru         5729
hi         5534
ko         4048
ta         3700
tl         3677
tr         3522
ml         3384
cn         3377
te         2713
ar         2550
nl         2202
id         2008
bn         1977
fa         1960
pl         1897
sv         1814
cs         1541
kn         1526
el         1368
th         1226
da         1221
he         1168
fi         1134
hu         1050
no         1002
Name: count, dtype: int64


In [79]:
final_df = final_df.loc[:, ~final_df.columns.duplicated()].copy()

print("Duplicate columns:", final_df.columns.duplicated().sum())

Duplicate columns: 0


## Fill some missing values

In [80]:
runtime_median = final_df["runtime"].median()

final_df["runtime"] = final_df["runtime"].fillna(runtime_median)

print("Median runtime:", runtime_median)
print("Missing runtime:", final_df["runtime"].isna().sum())

Median runtime: 92.0
Missing runtime: 0


## Creating dataset with only confirmed budget (Unfourtanately very unbalanced)

In [81]:
model_df = final_df.dropna(
    subset=["budget_2025"]
).copy()

print("Movies with budget:", len(model_df))

print(
    model_df["any_feature_film_nomination"]
    .value_counts()
)

positive_rate = model_df["any_feature_film_nomination"].mean()
print("Positive percentage:", positive_rate * 100)

Movies with budget: 26736
any_feature_film_nomination
0    26242
1      494
Name: count, dtype: int64
Positive percentage: 1.847695990424895


In [82]:
genre_cols = genre_dummies.columns.tolist()
language_cols = language_dummies.columns.tolist()

base_features = [
    "release_year",
    "release_month",
    "runtime"
] + genre_cols + language_cols

budget_features = [
    "release_year",
    "release_month",
    "runtime",
    "budget_2025"
] + genre_cols + language_cols

model_df = final_df.dropna(
    subset=["budget_2025"]
).copy()

X_base = model_df[base_features]
X_budget = model_df[budget_features]

y = model_df["any_feature_film_nomination"]

## Training


In [83]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_base_train = X_base.loc[train_idx]
X_base_test = X_base.loc[test_idx]

X_budget_train = X_budget.loc[train_idx]
X_budget_test = X_budget.loc[test_idx]

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

print("Training movies:", len(train_idx))
print("Testing movies:", len(test_idx))

Training movies: 21388
Testing movies: 5348


In [84]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

base_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=5000
    )
)

budget_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=5000
    )
)

base_model.fit(X_base_train, y_train)
budget_model.fit(X_budget_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](57,)","['release_year','release_month','runtime',...,'lang_tl','lang_tr', 'lang_zh']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,57
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [85]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

base_pred = base_model.predict(X_base_test)
base_prob = base_model.predict_proba(X_base_test)[:, 1]

budget_pred = budget_model.predict(X_budget_test)
budget_prob = budget_model.predict_proba(X_budget_test)[:, 1]

print("BASE MODEL")
print(classification_report(y_test, base_pred))
print("ROC-AUC:", roc_auc_score(y_test, base_prob))
print("PR-AUC:", average_precision_score(y_test, base_prob))

print("\nBUDGET MODEL")
print(classification_report(y_test, budget_pred))
print("ROC-AUC:", roc_auc_score(y_test, budget_prob))
print("PR-AUC:", average_precision_score(y_test, budget_prob))

BASE MODEL
              precision    recall  f1-score   support

           0       1.00      0.83      0.90      5249
           1       0.09      0.85      0.15        99

    accuracy                           0.83      5348
   macro avg       0.54      0.84      0.53      5348
weighted avg       0.98      0.83      0.89      5348

ROC-AUC: 0.9048957858254868
PR-AUC: 0.15651401118546357

BUDGET MODEL
              precision    recall  f1-score   support

           0       1.00      0.83      0.91      5249
           1       0.09      0.84      0.16        99

    accuracy                           0.83      5348
   macro avg       0.54      0.83      0.53      5348
weighted avg       0.98      0.83      0.89      5348

ROC-AUC: 0.9110460674568124
PR-AUC: 0.169444060018429


## Inspecting learned coefficients (How likely is to recieve a nomination. Higher coefficient higher chances)

In [86]:
import pandas as pd

log_reg = budget_model.named_steps["logisticregression"]

importance = pd.DataFrame({
    "feature": budget_features,
    "coefficient": log_reg.coef_[0]
})

importance = importance.sort_values(
    "coefficient",
    ascending=False
)

print(importance.head(15))
print("\nLowest coefficients:")
print(importance.tail(15))

          feature  coefficient
2         runtime     1.305655
10          Drama     0.858961
6       Animation     0.710420
31        lang_en     0.679884
9     Documentary     0.530148
35        lang_fr     0.375649
3     budget_2025     0.357825
1   release_month     0.357074
32        lang_es     0.316983
29        lang_de     0.219003
28        lang_da     0.197542
47        lang_pl     0.196848
21            War     0.177032
56        lang_zh     0.176097
46        lang_no     0.172335

Lowest coefficients:
         feature  coefficient
0   release_year    -0.344928
26       lang_cn    -0.347349
19      TV Movie    -0.348537
42       lang_kn    -0.403588
55       lang_tr    -0.428501
14        Horror    -0.457382
4         Action    -0.457522
53       lang_th    -0.471214
39       lang_id    -0.558407
52       lang_te    -0.605542
54       lang_tl    -0.611457
44       lang_ml    -0.636993
51       lang_ta    -0.642630
34       lang_fi    -0.732259
25       lang_bn    -3.666501


## Train models for all Oscar targets

In [87]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

target_names = [
    "any_feature_film_nomination",
    "any_feature_film_win",
    "best_picture_nomination",
    "best_picture_win",
    "animated_feature_nomination",
    "animated_feature_win",
    "international_feature_nomination",
    "international_feature_win",
    "documentary_feature_nomination",
    "documentary_feature_win"
]

oscar_models = {}

for target in target_names:
    y_target = model_df[target]

    X_train, X_test, y_train_target, y_test_target = train_test_split(
        model_df[budget_features],
        y_target,
        test_size=0.2,
        random_state=42,
        stratify=y_target
    )

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=5000
        )
    )

    model.fit(X_train, y_train_target)

    oscar_models[target] = model

    print(target, "trained")

any_feature_film_nomination trained
any_feature_film_win trained
best_picture_nomination trained
best_picture_win trained
animated_feature_nomination trained
animated_feature_win trained
international_feature_nomination trained
international_feature_win trained
documentary_feature_nomination trained
documentary_feature_win trained


In [88]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

nomination_targets = [
    "any_feature_film_nomination",
    "best_picture_nomination",
    "animated_feature_nomination",
    "international_feature_nomination",
    "documentary_feature_nomination"
]

calibrated_nomination_models = {}

for target in nomination_targets:

    y_target = model_df[target]

    X_train, X_test, y_train_target, y_test_target = train_test_split(
        model_df[budget_features],
        y_target,
        test_size=0.2,
        random_state=42,
        stratify=y_target
    )

    base_classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=5000
        )
    )

    calibrated_model = CalibratedClassifierCV(
        base_classifier,
        method="sigmoid",
        cv=3
    )

    calibrated_model.fit(
        X_train,
        y_train_target
    )

    calibrated_nomination_models[target] = calibrated_model

    print(target, "calibrated")

any_feature_film_nomination calibrated
best_picture_nomination calibrated
animated_feature_nomination calibrated
international_feature_nomination calibrated
documentary_feature_nomination calibrated


In [96]:
win_pairs = {
    "any_feature_film": (
        "any_feature_film_nomination",
        "any_feature_film_win"
    ),
    "best_picture": (
        "best_picture_nomination",
        "best_picture_win"
    ),
    "animated_feature": (
        "animated_feature_nomination",
        "animated_feature_win"
    ),
    "international_feature": (
        "international_feature_nomination",
        "international_feature_win"
    ),
    "documentary_feature": (
        "documentary_feature_nomination",
        "documentary_feature_win"
    )
}

conditional_win_models = {}
fallback_win_rates = {}

for name, (nomination_col, win_col) in win_pairs.items():

    nominated_movies = model_df[
        model_df[nomination_col] == 1
    ].copy()

    print(
        name,
        "- nominations:",
        len(nominated_movies),
        "- wins:",
        nominated_movies[win_col].sum()
    )

    # Too little data -> use historical win rate instead
    if len(nominated_movies) < 50:

        win_rate = nominated_movies[win_col].mean()

        fallback_win_rates[name] = win_rate

        print(
            "Using fallback win rate:",
            round(win_rate * 100, 1),
            "%"
        )

        continue

    X_win = nominated_movies[budget_features]
    y_win = nominated_movies[win_col]

    win_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=5000
        )
    )

    win_model.fit(
        X_win,
        y_win
    )

    conditional_win_models[name] = win_model

    print("Win model trained\n")

any_feature_film - nominations: 494 - wins: 111
Win model trained

best_picture - nominations: 293 - wins: 46
Win model trained

animated_feature - nominations: 94 - wins: 25
Win model trained

international_feature - nominations: 101 - wins: 31
Win model trained

documentary_feature - nominations: 23 - wins: 10
Using fallback win rate: 43.5 %


In [104]:
def predict_movie():
    print("Enter movie information.")
    print("Press Enter to skip any field.\n")

    title = input("Movie title: ").strip()

    year_input = input("Release year (1980-2025): ").strip()
    month_input = input("Release month (1-12): ").strip()
    runtime_input = input("Runtime in minutes: ").strip()
    budget_input = input("Budget in USD: ").replace(",", "").strip()
    genres_input = input(
        "Genres (example: Drama, History): "
    ).strip()
    language_input = input(
        "Original language code (en, fr, de, etc.): "
    ).strip().lower()

    # Start with all model features = 0
    movie = {
        feature: 0
        for feature in budget_features
    }

    # -------------------------
    # Basic numeric features
    # -------------------------

    release_year = (
        int(year_input)
        if year_input
        else int(model_df["release_year"].median())
    )

    movie["release_year"] = release_year

    movie["release_month"] = (
        int(month_input)
        if month_input
        else int(model_df["release_month"].median())
    )

    movie["runtime"] = (
        float(runtime_input)
        if runtime_input
        else float(model_df["runtime"].median())
    )

    # -------------------------
    # Budget
    # -------------------------

    if budget_input:
        budget = float(budget_input)
        entered_budget = budget

        # Convert historical budget to 2025 dollars
        if release_year in cpi:
            movie["budget_2025"] = (
                budget *
                (CPI_2025 / cpi[release_year])
            )
        else:
            movie["budget_2025"] = budget

    else:
        entered_budget = None

        movie["budget_2025"] = float(
            model_df["budget_2025"].median()
        )

    # -------------------------
    # Genres
    # -------------------------

    genre_lookup = {
        genre.lower(): genre
        for genre in genre_cols
    }

    if genres_input:
        entered_genres = [
            genre.strip().lower()
            for genre in genres_input.split(",")
        ]

        for genre in entered_genres:
            if genre in genre_lookup:
                real_column = genre_lookup[genre]
                movie[real_column] = 1

    # -------------------------
    # Language
    # -------------------------

    if language_input:
        language_column = f"lang_{language_input}"

        if language_column in language_cols:
            movie[language_column] = 1

        elif "lang_Other" in language_cols:
            movie["lang_Other"] = 1

    elif "lang_Other" in language_cols:
        movie["lang_Other"] = 1

    # -------------------------
    # DataFrame for models
    # -------------------------

    movie_df = pd.DataFrame(
        [movie],
        columns=budget_features
    )

    # Names we want to display
    display_names = {
        "any_feature_film_nomination":
            "Any feature-film nomination",

        "any_feature_film_win":
            "Any feature-film win",

        "best_picture_nomination":
            "Best Picture nomination",

        "best_picture_win":
            "Best Picture win",

        "animated_feature_nomination":
            "Animated Feature nomination",

        "animated_feature_win":
            "Animated Feature win",

        "international_feature_nomination":
            "International Feature nomination",

        "international_feature_win":
            "International Feature win",

        "documentary_feature_nomination":
            "Documentary Feature nomination",

        "documentary_feature_win":
            "Documentary Feature win"
    }

    # -------------------------
    # Predictions
    # -------------------------

    # Nomination probabilities
    nom_probs = {}

    for target, model in calibrated_nomination_models.items():
        nom_probs[target] = model.predict_proba(movie_df)[0, 1]


    # Make "any nomination" at least as large
    # as every individual category nomination
    category_nom_targets = [
        "best_picture_nomination",
        "animated_feature_nomination",
        "international_feature_nomination",
        "documentary_feature_nomination"
    ]

    nom_probs["any_feature_film_nomination"] = max(
        nom_probs["any_feature_film_nomination"],
        max(nom_probs[target] for target in category_nom_targets)
    )


    # -------------------------
    # Win probabilities
    # -------------------------

    win_probs = {}

    category_info = {
        "any_feature_film": (
            "any_feature_film_nomination",
            "any_feature_film_win"
        ),

        "best_picture": (
            "best_picture_nomination",
            "best_picture_win"
        ),

        "animated_feature": (
            "animated_feature_nomination",
            "animated_feature_win"
        ),

        "international_feature": (
            "international_feature_nomination",
            "international_feature_win"
        ),

        "documentary_feature": (
            "documentary_feature_nomination",
            "documentary_feature_win"
        )
    }


    for name, (nom_target, win_target) in category_info.items():

        nomination_probability = nom_probs[nom_target]

        # ML win model available
        if name in conditional_win_models:

            conditional_win_probability = (
                conditional_win_models[name]
                .predict_proba(movie_df)[0, 1]
            )

        # Small dataset -> historical fallback rate
        else:

            conditional_win_probability = (
                fallback_win_rates[name]
            )

        # P(win) = P(nomination) × P(win | nominated)
        win_probs[win_target] = (
            nomination_probability *
            conditional_win_probability
        )


    # "Any win" must be at least as large
    # as each individual category win
    category_win_targets = [
        "best_picture_win",
        "animated_feature_win",
        "international_feature_win",
        "documentary_feature_win"
    ]

    win_probs["any_feature_film_win"] = max(
        win_probs["any_feature_film_win"],
        max(win_probs[target] for target in category_win_targets)
    )


    # -------------------------
    # Display
    # -------------------------

    print("\n----------------------------------")
    print(title if title else "Unnamed Movie")
    print("----------------------------------")

    print("Input values:")
    print(f"Release year:      {release_year}")
    print(f"Release month:     {movie['release_month']}")
    print(f"Runtime:           {movie['runtime']} min")

    if entered_budget is not None:
        print(f"Entered budget:    ${entered_budget:,.0f}")
    else:
        print("Entered budget:    skipped")
    
    print(f"Budget (2025 USD): ${movie['budget_2025']:,.0f}")

    # Show entered genres
    selected_genres = [
        genre
        for genre in genre_cols
        if movie.get(genre, 0) == 1
    ]

    print(
        "Genres:            "
        + (", ".join(selected_genres) if selected_genres else "None")
    )

    # Show selected language
    selected_language = None

    for lang_col in language_cols:
        if movie.get(lang_col, 0) == 1:
            selected_language = lang_col.replace("lang_", "")
            break

    print(
        "Language:          "
        + (selected_language if selected_language else "None")
    )

    print("\nPredictions:")

    print(
        f"{'Any feature-film nomination':38s}"
        f"{nom_probs['any_feature_film_nomination'] * 100:6.1f}%"
    )

    print(
        f"{'Any feature-film win':38s}"
        f"{win_probs['any_feature_film_win'] * 100:6.1f}%"
    )

    print()

    print(
        f"{'Best Picture nomination':38s}"
        f"{nom_probs['best_picture_nomination'] * 100:6.1f}%"
    )

    print(
        f"{'Best Picture win':38s}"
        f"{win_probs['best_picture_win'] * 100:6.1f}%"
    )

    print()

    print(
        f"{'Animated Feature nomination':38s}"
        f"{nom_probs['animated_feature_nomination'] * 100:6.1f}%"
    )

    print(
        f"{'Animated Feature win':38s}"
        f"{win_probs['animated_feature_win'] * 100:6.1f}%"
    )

    print()

    print(
        f"{'International Feature nomination':38s}"
        f"{nom_probs['international_feature_nomination'] * 100:6.1f}%"
    )

    print(
        f"{'International Feature win':38s}"
        f"{win_probs['international_feature_win'] * 100:6.1f}%"
    )

    print()

    print(
        f"{'Documentary Feature nomination':38s}"
        f"{nom_probs['documentary_feature_nomination'] * 100:6.1f}%"
    )

    print(
        f"{'Documentary Feature win':38s}"
        f"{win_probs['documentary_feature_win'] * 100:6.1f}%"
    )

In [103]:
predict_movie()

Enter movie information.
Press Enter to skip any field.


----------------------------------
Example
----------------------------------
Input values:
Release year:      2024
Release month:     11
Runtime:           138.0 min
Entered budget: $300,000,000
Budget (2025 USD): $307,893,806
Genres:            Drama
Language:          en

Predictions:
Any feature-film nomination             19.1%
Any feature-film win                     4.0%

Best Picture nomination                 17.8%
Best Picture win                         0.6%

Animated Feature nomination              1.0%
Animated Feature win                     0.9%

International Feature nomination         0.2%
International Feature win                0.2%

Documentary Feature nomination           0.2%
Documentary Feature win                  0.1%
